# Online Retail Analysis

This notebook reproduces the five Excel tasks in Python (pandas / SciPy) so the whole
workflow is reproducible and version-controlled.

**Before running:** place `OnlineRetail.csv` in the same folder as this notebook.
The classic UCI "Online Retail" dataset has these columns:

`InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country`

The notebook also writes out an `OnlineRetail_Analysis.xlsx` workbook at the end, with one
sheet per task, so you have an Excel-friendly copy of every result as well.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

CSV_PATH = "OnlineRetail.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"'{CSV_PATH}' not found in the working directory. "
        "Place the OnlineRetail.csv file next to this notebook and re-run."
    )

# The classic UCI Online Retail file is typically saved with Latin-1 (ISO-8859-1) encoding.
try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, encoding="ISO-8859-1")

# Basic cleanup: standardize column names in case of stray whitespace
df.columns = [c.strip() for c in df.columns]

# Parse dates and numeric columns
if "InvoiceDate" in df.columns:
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

print(df.shape)
df.head()


## Task 1 — Filter UK transactions & count unique customers

Equivalent of applying the Excel Filter to `Country = "United Kingdom"` and counting
unique `CustomerID` values.


In [ ]:
uk_df = df[df["Country"].astype(str).str.strip() == "United Kingdom"].copy()

uk_unique_customers = uk_df["CustomerID"].dropna().nunique()

print(f"UK transactions: {len(uk_df):,}")
print(f"Unique UK CustomerIDs: {uk_unique_customers:,}")

uk_df.head()


## Task 2 — Total sales by country (PivotTable equivalent)

`Sales = Quantity * UnitPrice`, summed by `Country`. Top 3 countries by total sales are
identified below.


In [ ]:
df["Sales"] = df["Quantity"] * df["UnitPrice"]

pivot_country = (
    df.groupby("Country", dropna=False)["Sales"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Sales": "TotalSales"})
)

top3_countries = pivot_country.head(3)["Country"].tolist()

print("Top 3 countries by total sales:", top3_countries)
pivot_country.head(10)


## Task 3 — Average order value & highlighting orders above ₹10,000

Order value is total `Sales` per `InvoiceNo`. We compute the average order value across
all invoices, then flag (the Python equivalent of Conditional Formatting) every invoice
whose value exceeds ₹10,000.


In [ ]:
order_value = (
    df.groupby("InvoiceNo", dropna=False)["Sales"]
      .sum()
      .reset_index()
      .rename(columns={"Sales": "OrderValue"})
)

average_order_value = order_value["OrderValue"].mean()
print(f"Average order value (all invoices): {average_order_value:,.2f}")

THRESHOLD = 10000
order_value["AboveThreshold"] = order_value["OrderValue"] > THRESHOLD

high_value_orders = order_value[order_value["AboveThreshold"]].sort_values(
    "OrderValue", ascending=False
)
print(f"Invoices above ₹{THRESHOLD:,}: {len(high_value_orders):,}")

# Visual highlight inline in the notebook (equivalent of conditional formatting)
def highlight_above(row):
    return ["background-color: #ffcccc" if row["AboveThreshold"] else "" for _ in row]

order_value.sort_values("OrderValue", ascending=False).head(20).style.apply(
    highlight_above, axis=1
)


## Task 4 — t-test: France vs Germany average order value

Welch's t-test (two-sample, unequal variances) — the direct equivalent of the Data
Analysis Toolpak's "t-Test: Two-Sample Assuming Unequal Variances".


In [ ]:
# Order value restricted to invoices whose (first-seen) country is France / Germany
invoice_country = df.groupby("InvoiceNo")["Country"].first()
order_value_c = order_value.merge(
    invoice_country.rename("Country"), left_on="InvoiceNo", right_index=True
)

france_values = order_value_c.loc[order_value_c["Country"] == "France", "OrderValue"].dropna()
germany_values = order_value_c.loc[order_value_c["Country"] == "Germany", "OrderValue"].dropna()

t_stat, p_value = stats.ttest_ind(france_values, germany_values, equal_var=False)

alpha = 0.05
significant = p_value < alpha

print(f"France: n={len(france_values)}, mean={france_values.mean():,.2f}")
print(f"Germany: n={len(germany_values)}, mean={germany_values.mean():,.2f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
print(
    f"Result: {'Statistically significant' if significant else 'Not statistically significant'} "
    f"difference at the 0.05 level."
)


## Task 5 — Insights summary

A written summary of the key patterns observed above (in place of a ChatGPT/Copilot
call, since this notebook doesn't have an external AI API wired in). Re-run the cells
above first so the f-string below reflects your actual data.


In [ ]:
insights = f'''
Key Customer Purchasing Patterns
=================================

1. Market concentration: {top3_countries[0]} leads total sales, followed by
   {top3_countries[1]} and {top3_countries[2]}. The UK alone accounts for
   {uk_unique_customers:,} unique customers, underlining its role as the core market.

2. Average order size: across all invoices, the average order value is
   {average_order_value:,.2f}. A subset of {len(high_value_orders):,} invoices exceed
   the {THRESHOLD:,} high-value threshold, suggesting a small group of large/bulk
   orders (likely wholesale buyers) pulls the average up relative to typical
   transactions.

3. France vs Germany: the average order value is {france_values.mean():,.2f} for
   France and {germany_values.mean():,.2f} for Germany. The difference is
   {'statistically significant' if significant else 'not statistically significant'}
   at the 0.05 level (p = {p_value:.4f}), so {'these two markets behave differently in typical order size' if significant else 'order sizes in these two markets look comparable once sample variation is accounted for'}.

4. Recommended follow-ups: segment high-value orders by product category to see what
   drives bulk purchases, and examine InvoiceDate for seasonal peaks (e.g. Nov/Dec
   holiday spikes) to inform inventory and marketing timing.
'''

print(insights)


## Export everything to Excel

Writes one workbook, `OnlineRetail_Analysis.xlsx`, with a sheet per task (including an
`Insights` sheet holding the summary text above) so you have an Excel-ready copy of
every result.


In [ ]:
with pd.ExcelWriter("OnlineRetail_Analysis.xlsx", engine="openpyxl") as writer:
    uk_df.to_excel(writer, sheet_name="UK_Filtered", index=False)
    pivot_country.to_excel(writer, sheet_name="Sales_By_Country", index=False)
    order_value.sort_values("OrderValue", ascending=False).to_excel(
        writer, sheet_name="Order_Value", index=False
    )
    pd.DataFrame({"Insight": [insights]}).to_excel(
        writer, sheet_name="Insights", index=False
    )

# Conditional formatting (highlight OrderValue > threshold) + top-3-country note
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.formatting.rule import CellIsRule

wb = load_workbook("OnlineRetail_Analysis.xlsx")

ws = wb["Order_Value"]
max_row = ws.max_row
red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
# OrderValue is column B (InvoiceNo=A, OrderValue=B, AboveThreshold=C)
ws.conditional_formatting.add(
    f"B2:B{max_row}",
    CellIsRule(operator="greaterThan", formula=[str(THRESHOLD)], fill=red_fill),
)

ws2 = wb["Sales_By_Country"]
ws2[f"A{ws2.max_row + 2}"] = "Top 3 countries:"
ws2[f"A{ws2.max_row + 1}"] = ", ".join(top3_countries)

wb.save("OnlineRetail_Analysis.xlsx")
print("Saved OnlineRetail_Analysis.xlsx")
